In [ ]:
import os
import pandas as pd
from uqlm import WhiteBoxUQ
from uqlm.utils import load_example_dataset
from langchain_together import ChatTogether
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
rate_limiter = InMemoryRateLimiter(
    requests_per_second=4.5, check_every_n_seconds=0.5, max_bucket_size=280,  
)
from utils import get_datasets, get_models

In [ ]:
from uqlm.utils.prompts.codegen import python_prompt_template, python_prompt_template_stdio
from uqlm.code.code_evaluation import evaluate_python_code

lcb = load_example_dataset("livecodebench")

# prompts = []
# for i in range(len(df)):
#     # For callable-style questions
#     if df["platform"].iloc[i] == "leetcode":
#         prompt = python_prompt_template(df["question_content"].iloc[i], df["starter_code"].iloc[i])

#     # For I/O style questions
#     else:
#         prompt = python_prompt_template_stdio(df["question_content"].iloc[i])
#     prompts.append(prompt)



In [ ]:
datasets = get_datasets()
models = get_models()
models.keys()

In [ ]:
import numpy as np
for model in models:
    if model == "gpt-4o-mini":
        continue
    path = f"../data/output_data/livecodebench-{model}.parquet"
    df = pd.read_parquet(path)
    

    responses = df["response"].to_list()
    public_test_cases = lcb["public_test_cases"].to_list()
    metadata = lcb["metadata"].to_list()

    # Use uqlm python grader utility `evaluate_python_code` to evaluate whether unit tests pass
    result_dict = evaluate_python_code(responses=responses, public_test_cases=public_test_cases, metadata=metadata)

    #  Convert results to DataFrame
    eval_df = pd.DataFrame(result_dict)
    df["correct"] = eval_df["unit_test_passed"]
    print(model, np.mean(df.correct))
    df.to_parquet(path)

In [ ]:
import numpy as np
np.mean(df.correct)

In [ ]:
# import os
# import time
# from pathlib import Path

# BATCH_SIZE = 180
# OUTPUT_DIR = Path("../data/output_data")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# scorers = ["sequence_probability", "min_probability", "min_token_negentropy", "mean_token_negentropy", "probability_margin"]
# for model_name, llm in models.items():
#     # for dataset_name, df in datasets.items():
#     for dataset_name in ["livecodebench"]:
#         if ('deepseek' in model_name) or ('llama' in model_name) or ('qwen' in model_name) or ("ini" in model_name):
#             continue


#         # prompts = df["prompt"].tolist()
#         n_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

#         print(f"\n{dataset_name} | {model_name} — {len(prompts)} prompts, {n_batches} batches")

#         for i in range(n_batches):
#             try: 
#                 batch_path = OUTPUT_DIR / f"{dataset_name}-{model_name}-batch-{i:04d}.parquet"

#                 if batch_path.exists():
#                     print(f"  batch {i:04d}: already done, skipping")
#                     continue

#                 start, end = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, len(prompts))
#                 batch_prompts = prompts[start:end]
#                 # batch_gt = df["ground_truth"].iloc[start:end].tolist()

#                 print(f"  batch {i:04d}: rows {start}–{end-1} ...", end=" ", flush=True)
#                 top_k_logprobs = 5 if (("llama" in model_name) or ("qwen" in model_name)) else 15
#                 max_calls_per_min = 15 

#                 wbuq = WhiteBoxUQ(
#                     llm=llm, 
#                     scorers=scorers, 
#                     max_calls_per_min=max_calls_per_min, 
#                     top_k_logprobs=top_k_logprobs
#                 )
#                 result = await wbuq.generate_and_score(prompts=batch_prompts)
#                 batch_df = result.to_df()
#                 # batch_df["ground_truth"] = batch_gt
#                 batch_df.to_parquet(batch_path, index=False)
#                 print(f"saved {batch_path.name}")
#                 time.sleep(61)
#             except Exception as e:
#                 print(e)
#                 time.sleep(61)
#                 print("retrying....")
#                 pass


In [ ]:
# Merge all completed batches into a single parquet per (dataset, model)
import re
import os
from pathlib import Path
OUTPUT_DIR = Path("../data/output_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for model_name in models:
    # for dataset_name, df in datasets.items():
    for dataset_name in ["livecodebench"]:

        batch_files = sorted(OUTPUT_DIR.glob(f"{dataset_name}-{model_name}-batch-*.parquet"))
        if not batch_files:
            continue
        merged = pd.concat([pd.read_parquet(f) for f in batch_files], ignore_index=True)
        out_path = OUTPUT_DIR / f"{dataset_name}-{model_name}.parquet"
        merged.to_parquet(out_path, index=False)
        print(f"{dataset_name}-{model_name}: merged {len(batch_files)} batches → {len(merged)} rows → {out_path.name}")


In [ ]:
datasets = get_datasets()
datasets.keys()
df2 = datasets["math_hard"]
df1 = merged

In [ ]:
import pandas as pd
import numpy as np

# Get the unique lookup values from df2 in their original order
lookup_values = df2["prompt"].unique()

# Chunk df2's unique values into groups of 50
chunks = [lookup_values[i:i + 50] for i in range(0, len(lookup_values), 50)]

# For each chunk, filter df1 and write to parquet
for idx, chunk in enumerate(chunks):
    subset = df1[df1["prompt"].isin(chunk)]
    if not subset.empty:
        subset.to_parquet(f"math_hard-deepseek-v3-batch-{idx:04d}.parquet", index=False)

In [ ]:
tmp = pd.read_parquet("../data/output_data/mmlu-gemma-3n-e4b.parquet")